# Support Vector Machines Assignment

Analysis of the famous **Iris flower dataset** using SVM classification.

## The Data

The Iris dataset contains 150 samples across 3 species (setosa, versicolor, virginica),
with 4 features each: sepal length, sepal width, petal length, petal width.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.datasets import load_iris

## Get the Data

In [1]:
# Load via sklearn (equivalent to sns.load_dataset('iris'))
iris_sk = load_iris()
iris = pd.DataFrame(iris_sk.data, columns=['sepal_length','sepal_width','petal_length','petal_width'])
iris['species'] = pd.Categorical.from_codes(iris_sk.target, iris_sk.target_names)
iris.head()

## Exploratory Data Analysis

### Pairplot — Which species is most separable?

**Iris setosa** is clearly the most linearly separable species — it forms a distinct cluster from versicolor and virginica in almost every feature combination. Versicolor and virginica overlap more, especially in sepal dimensions.

In [1]:
sns.pairplot(iris, hue='species', palette='Set2')
plt.suptitle('Iris Dataset — Pairplot by Species', y=1.02)
plt.show()

### KDE Plot — Sepal Length vs Sepal Width for Setosa

In [1]:
setosa = iris[iris['species'] == 'setosa']
plt.figure(figsize=(8, 6))
sns.kdeplot(data=setosa, x='sepal_length', y='sepal_width', fill=True, cmap='Blues')
plt.title('KDE — Sepal Length vs Sepal Width (Iris Setosa)')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.show()

## Train Test Split

In [1]:
from sklearn.model_selection import train_test_split

X = iris.drop('species', axis=1)
y = iris['species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101)
print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

Training set: (105, 4)
Test set:     (45, 4)


## Train a Support Vector Machine Classifier

In [1]:
from sklearn.svm import SVC

model = SVC()
model.fit(X_train, y_train)

## Model Evaluation

In [1]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print()
print("Classification Report:")
print(classification_report(y_test, predictions))

Confusion Matrix:
[[13  0  0]
 [ 0 19  1]
 [ 0  0 12]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        13
  versicolor       1.00      0.95      0.97        20
   virginica       0.92      1.00      0.96        12

    accuracy                           0.98        45
   macro avg       0.97      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45


The initial SVC already achieves **98% accuracy** on the test set — only 1 misclassification (a virginica predicted as versicolor). The Iris dataset is relatively easy for SVM.

## Gridsearch Practice

Let's tune hyperparameters `C` and `gamma` to see if we can improve further.

In [1]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001], 'kernel': ['rbf']}
grid = GridSearchCV(SVC(), param_grid, cv=5, verbose=1)
grid.fit(X_train, y_train)

In [1]:
print("Best parameters:", grid.best_params_)
print("Best estimator:", grid.best_estimator_)

Best parameters: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Best estimator: SVC(gamma=0.1)


In [1]:
grid_predictions = grid.predict(X_test)

print("Confusion Matrix (GridSearch):")
print(confusion_matrix(y_test, grid_predictions))
print()
print("Classification Report (GridSearch):")
print(classification_report(y_test, grid_predictions))

Confusion Matrix (GridSearch):
[[13  0  0]
 [ 0 19  1]
 [ 0  0 12]]

Classification Report (GridSearch):
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        13
  versicolor       1.00      0.95      0.97        20
   virginica       0.92      1.00      0.96        12

    accuracy                           0.98        45
   macro avg       0.97      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45


## Results Summary

| Model | Accuracy | Misclassifications |
|---|---|---|
| Default SVC | 98% | 1 |
| GridSearch SVC (C=1, γ=0.1) | 98% | 1 |

**Were we able to improve?** No further improvement was achieved — and that's expected.  
The Iris dataset is nearly linearly separable (especially setosa), so the default SVC already performs near-optimally.  
The GridSearch confirmed that `C=1, gamma=0.1` are the best RBF parameters, matching the default behavior closely.  
The single misclassification (a virginica classified as versicolor) reflects genuine overlap between those two species that even a well-tuned SVM cannot eliminate.